In [46]:
import pandas as pd
from pathlib import Path

In [47]:
print(list(Path("..").rglob("raw_data*.csv")))

[WindowsPath('../data/raw/raw_data.csv'), WindowsPath('../data/raw/raw_dataF.csv'), WindowsPath('../data/raw/raw_dataFF.csv')]


In [48]:
DATA_PATH = Path("..")  / "data" / "raw" / "raw_dataFF.csv"
df = pd.read_csv(DATA_PATH)

In [49]:
df.head()

,host_received_timestamp_utc,device_time_ms,stage_elapsed_ms,condition,motor_running,motor_direction,speed_percent,pwm_duty_8bit,ina219_ok,power_data_valid,...,gyro_y_rad_s,gyro_z_rad_s,encoder_count,encoder_delta,encoder_counts_per_second,encoder_counts_per_revolution,rpm_valid,rpm,overcurrent_fault,last_motor_stop_reason
0,2026-10-28T04:14:13.983+00:00,74978,200,normal,0,forward,0,0,1,1,...,-0.0304,-0.0061,0,0,0.0,5760,1,0.0,0,keyboard_stop
1,2026-10-28T04:14:14.183+00:00,75178,400,normal,0,forward,0,0,1,1,...,-0.0288,-0.0051,0,0,0.0,5760,1,0.0,0,keyboard_stop
2,2026-10-28T04:14:14.383+00:00,75378,600,normal,0,forward,0,0,1,1,...,-0.0280,-0.0056,0,0,0.0,5760,1,0.0,0,keyboard_stop
3,2026-10-28T04:14:14.583+00:00,75578,800,normal,0,forward,0,0,1,1,...,-0.0288,-0.0067,0,0,0.0,5760,1,0.0,0,keyboard_stop
4,2026-10-28T04:14:14.783+00:00,75778,1000,normal,0,forward,0,0,1,1,...,-0.0290,-0.0064,0,0,0.0,5760,1,0.0,0,keyboard_stop


In [50]:
df.columns

Index(['host_received_timestamp_utc', 'device_time_ms', 'stage_elapsed_ms',
       'condition', 'motor_running', 'motor_direction', 'speed_percent',
       'pwm_duty_8bit', 'ina219_ok', 'power_data_valid', 'bus_voltage_v',
       'shunt_voltage_mv', 'current_ma', 'power_mw', 'ds18b20_ok',
       'temperature_valid', 'temperature_c', 'mpu6050_ok', 'mpu_data_valid',
       'accel_x_m_s2', 'accel_y_m_s2', 'accel_z_m_s2', 'gyro_x_rad_s',
       'gyro_y_rad_s', 'gyro_z_rad_s', 'encoder_count', 'encoder_delta',
       'encoder_counts_per_second', 'encoder_counts_per_revolution',
       'rpm_valid', 'rpm', 'overcurrent_fault', 'last_motor_stop_reason'],
      dtype='object')

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20101 entries, 0 to 20100
Data columns (total 33 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   host_received_timestamp_utc    20101 non-null  object 
 1   device_time_ms                 20101 non-null  int64  
 2   stage_elapsed_ms               20101 non-null  int64  
 3   condition                      20101 non-null  object 
 4   motor_running                  20101 non-null  int64  
 5   motor_direction                20101 non-null  object 
 6   speed_percent                  20101 non-null  int64  
 7   pwm_duty_8bit                  20101 non-null  int64  
 8   ina219_ok                      20101 non-null  int64  
 9   power_data_valid               20101 non-null  int64  
 10  bus_voltage_v                  20101 non-null  float64
 11  shunt_voltage_mv               20101 non-null  float64
 12  current_ma                     20101 non-null 

In [52]:
df['host_received_timestamp_utc'].isna().sum()

np.int64(0)

In [53]:
df.isnull().sum().sort_values(ascending=False).head(5)

temperature_c                  30
host_received_timestamp_utc     0
device_time_ms                  0
condition                       0
stage_elapsed_ms                0
dtype: int64

In [54]:
df.duplicated().sum()

np.int64(450)

In [55]:
df["condition"].value_counts()

condition
normal             13870
overheated          4993
vibration_issue     1238
Name: count, dtype: int64

In [56]:
REQUIRED_COLUMNS = [
    "host_received_timestamp_utc", "condition", "motor_running",
    "speed_percent", "current_ma", "temperature_c",
    "accel_x_m_s2", "accel_y_m_s2", "accel_z_m_s2", "rpm"
]
VALID_CONDITIONS = {"normal", "overheated", "vibration_issue"}
REFINE_LABELS = True
HOT_TEMP_C = 30.0

In [57]:
df["host_received_timestamp_utc"] = pd.to_datetime(
        df["host_received_timestamp_utc"], errors="coerce", utc=True
    )

In [58]:
df["host_received_timestamp_utc"].head()

0   2026-10-28 04:14:13.983000+00:00
1   2026-10-28 04:14:14.183000+00:00
2   2026-10-28 04:14:14.383000+00:00
3   2026-10-28 04:14:14.583000+00:00
4   2026-10-28 04:14:14.783000+00:00
Name: host_received_timestamp_utc, dtype: datetime64[ns, UTC]

In [59]:
df['condition'] = df['condition'].str.lower().str.strip()
df["condition"].sample(5)

209          normal
19444        normal
4835     overheated
3118     overheated
5181     overheated
Name: condition, dtype: object

In [60]:
df['label'] = df['condition'].map({"normal": 0, "overheated": 1, "vibration_issue": 1}).astype('Int64')
df["label"].sample(5)

16256    0
16999    0
3439     1
8284     0
18112    0
Name: label, dtype: Int64

In [61]:
# find nan values in columns
for col in df.columns:
    if df[col].isnull().any():
        print(f"Column '{col}' has {df[col].isnull().sum()} NaN values.")

Column 'temperature_c' has 30 NaN values.


In [62]:
df['temperature_valid'].value_counts()

temperature_valid
1    20071
0       30
Name: count, dtype: int64

In [63]:
df['temperature_c'].unique()

array([26.5 , 26.75, 27.5 , 27.75, 28.  , 28.25, 28.5 , 28.75, 29.  ,
       29.25, 29.5 , 29.75, 30.5 , 30.75, 31.  , 31.25, 31.5 , 31.75,
       32.  , 32.25, 32.5 , 32.75,   nan, 33.  , 33.25, 33.5 , 33.75,
       34.  , 34.25, 24.5 , 24.75, 25.  , 25.25, 26.25, 30.25, 37.  ,
       40.75, 45.  , 50.  , 55.  , 60.25, 64.75, 68.  , 70.5 , 72.25,
       73.5 , 53.75, 74.75, 74.25, 74.  , 71.75, 71.25, 70.  , 69.25,
       68.75, 65.75, 64.25, 63.  , 61.5 , 59.25, 58.25, 57.25, 56.25,
       55.5 , 54.75, 54.  , 53.25, 52.5 , 51.75, 51.25, 50.5 , 49.5 ,
       48.75, 47.75, 47.  , 46.25, 45.5 , 44.75, 44.  , 43.5 , 43.  ,
       40.5 , 40.25, 40.  , 39.75, 39.25, 39.  , 38.75, 38.5 , 38.25,
       38.  , 36.  , 35.75, 35.5 , 35.25, 34.5 , 81.5 , 81.  , 80.5 ,
       79.75, 79.  , 78.25, 77.5 , 76.5 , 75.75, 75.25, 75.5 , 77.25,
       77.75, 78.  , 78.75, 80.75, 81.25, 82.25, 82.75, 83.  , 81.75,
       80.25, 79.5 , 76.75, 76.  , 72.5 , 67.5 , 66.5 , 65.25, 63.25,
       62.25, 61.25,

In [64]:
if "temperature_vaild" in df.columns:
    invalid_temperature = df["temperature_valid"].ne(1)
    df.loc[invalid_temperature, "temperature_c"] = pd.NA
    

In [65]:
df['temperature_c'].unique()

array([26.5 , 26.75, 27.5 , 27.75, 28.  , 28.25, 28.5 , 28.75, 29.  ,
       29.25, 29.5 , 29.75, 30.5 , 30.75, 31.  , 31.25, 31.5 , 31.75,
       32.  , 32.25, 32.5 , 32.75,   nan, 33.  , 33.25, 33.5 , 33.75,
       34.  , 34.25, 24.5 , 24.75, 25.  , 25.25, 26.25, 30.25, 37.  ,
       40.75, 45.  , 50.  , 55.  , 60.25, 64.75, 68.  , 70.5 , 72.25,
       73.5 , 53.75, 74.75, 74.25, 74.  , 71.75, 71.25, 70.  , 69.25,
       68.75, 65.75, 64.25, 63.  , 61.5 , 59.25, 58.25, 57.25, 56.25,
       55.5 , 54.75, 54.  , 53.25, 52.5 , 51.75, 51.25, 50.5 , 49.5 ,
       48.75, 47.75, 47.  , 46.25, 45.5 , 44.75, 44.  , 43.5 , 43.  ,
       40.5 , 40.25, 40.  , 39.75, 39.25, 39.  , 38.75, 38.5 , 38.25,
       38.  , 36.  , 35.75, 35.5 , 35.25, 34.5 , 81.5 , 81.  , 80.5 ,
       79.75, 79.  , 78.25, 77.5 , 76.5 , 75.75, 75.25, 75.5 , 77.25,
       77.75, 78.  , 78.75, 80.75, 81.25, 82.25, 82.75, 83.  , 81.75,
       80.25, 79.5 , 76.75, 76.  , 72.5 , 67.5 , 66.5 , 65.25, 63.25,
       62.25, 61.25,

In [66]:
df = df.sort_values(
        "host_received_timestamp_utc", kind="stable"
    ).reset_index(drop=True)

In [67]:
if REFINE_LABELS:
        temp = pd.to_numeric(df["temperature_c"], errors="coerce")
        # missing temperature on an overheated row: cannot tell, so it is left out too
        not_hot = (df["condition"] == "overheated") & ~(temp >= HOT_TEMP_C)
        idle_vibration = (df["condition"] == "vibration_issue") & (df["motor_running"] == 0)
        df.loc[not_hot, "label_note"] = "not_hot_yet"
        df.loc[idle_vibration, "label_note"] = "idle_motor"
        df.loc[not_hot | idle_vibration, "label"] = pd.NA


In [68]:
df['label_note'].value_counts()

label_note
not_hot_yet    2201
idle_motor      277
Name: count, dtype: int64

In [69]:
df["label"].unique()

<IntegerArray>
[0, <NA>, 1]
Length: 3, dtype: Int64

In [70]:
print(f"Rows: {len(df)}")
print("\nRecorded condition counts:")
print(df["condition"].value_counts(dropna=False).to_string())
print("\nLabel counts (NA = left out):")
print(df["label"].value_counts(dropna=False).to_string())
print("\nMissing values:")
print(df.isna().sum().loc[lambda values: values > 0].to_string())
    

Rows: 20101

Recorded condition counts:
condition
normal             13870
overheated          4993
vibration_issue     1238

Label counts (NA = left out):
label
0       13870
1        3753
<NA>     2478

Missing values:
temperature_c       30
label             2478
label_note       17623
